# Retrieval of MS MARCO v1 passages based on BM25 via OpenSearch

MS MARCO passages are flat and title-less (`docid`, `text` only), so both the
query and the display differ from the title+text notebook -- hence a separate
notebook per the one-structure-per-notebook convention.

#### Configuration

In [ ]:
index_name = "msmarco_v1_passage_bm25"
q = "do goldfish grow"
dataset_name = "msmarco-passage/trec-dl-2019/judged"

In [ ]:
import sys
!{sys.executable} -m pip install -q ir_datasets opensearch-py dotenv

In [ ]:
import pprint

Your opensearch password should be available in `~/.env`

```bash
    OPENSEARCH_INITIAL_ADMIN_PASSWORD="strong password"
```

In [ ]:
import os
from dotenv import load_dotenv
from opensearchpy import OpenSearch

load_dotenv()
host = 'localhost'
port = 9200
password = os.getenv("OPENSEARCH_INITIAL_ADMIN_PASSWORD")

client = OpenSearch(
    hosts=[{"host": host, "port": port}],
    http_auth=("admin", password),
    http_compress=True,
    use_ssl=True,
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)
pprint.pprint(client.info())

#### BM25 Search

In [ ]:
def build_query(query: str) -> dict:
    return {"match": {"text": query}}

def search(query: str, size: int = 10) -> dict:
    return client.search(index=index_name, body={
        "size": size,
        "_source": ["docid", "text"],
        "query": build_query(query),
    })

def show(resp, label=""):
    hits = resp["hits"]["hits"]
    print(f"\nTop {len(hits)} hits{' (' + label + ')' if label else ''}\n")
    for hit in hits:
        src = hit["_source"]
        print(f"[{src['docid']}] {src['text'][:80]}... (score={hit['_score']:.2f})")

In [ ]:
show(search(q, size=5), q)

#### Search with a Topic from the Dataset

In [ ]:
import ir_datasets
dataset = ir_datasets.load(dataset_name)

In [ ]:
topic = next(dataset.queries_iter())
pprint.pprint(topic)
topic_query = topic.text   # judged TREC DL topics are GenericQuery (query_id, text)
show(search(topic_query, size=5), f"topic {topic.query_id}: {topic_query}")

#### Rerank with the cross-encoder (common second stage)

Re-run the same first-stage query through the `rerank_bge_m3` search pipeline
(`BAAI/bge-reranker-v2-m3`, multilingual -- see
[ml_model_registration.ipynb](../indexing/opensearch/ml_model_registration.ipynb)).
Works identically over every index and ranker.

**Gotcha:** the request must return `_source` including the `text` field --
the rerank processor reads `document_fields: ["text"]` from `_source`; without
it every hit gets the same score and the order silently stays unchanged.

In [ ]:
def search_reranked(query: str, size: int = 10, rerank_pipeline: str = "rerank_bge_m3") -> dict:
    """Same first stage as search(), then cross-encoder reranking server-side."""
    return client.search(
        index=index_name,
        params={"search_pipeline": rerank_pipeline},
        body={
            "size": size,
            "_source": ["docid", "title", "text"],   # MUST include "text" (rerank context)
            "query": build_query(query),
            "ext": {"rerank": {"query_context": {"query_text": query}}},
        },
    )

show(search(q, size=5), "first stage")
show(search_reranked(q, size=5), "reranked")